# 📋 DQ Framework — Cell 1: Config Validator

**Purpose:** Defines all config loading and validation logic as plain Python functions.
This notebook is `%run` by the controller. It contains **no** imports of local modules.

### What gets defined here
| Function | Purpose |
|---|---|
| `load_master_config()` | Parse master_config.yml, check duplicates |
| `load_dataset_config()` | Parse one dataset YAML, validate rules |
| `load_all_configs()` | Orchestrate full config pre-flight validation |
| `_validate_rule()` | Validate a single rule dict (all 16 types) |

### Supported rule types (16)
`not_null` · `unique` · `row_count` · `accepted_values` · `regex` · `range` ·
`date_range` · `referential` · `freshness` · `custom_sql` · `completeness` ·
`duplicate_count` · `conditional_not_null` · `mutual_exclusivity` ·
`character_set` · `length_check`

## Imports

In [0]:
import os
import yaml
from pathlib import Path
from typing import Any

## Custom Exceptions

In [0]:
class DQConfigError(Exception):
    """Raised when master or dataset config validation fails. Aborts the entire run."""
    pass


class DQDatasetConfigError(DQConfigError):
    """Raised for errors in a specific dataset config file."""
    pass

## Constants — Valid Rule Types and Severities

In [0]:
VALID_RULE_TYPES = {
    # ── Original 12 ────────────────────────────────────────────────────────
    "not_null",           # Column must have zero nulls
    "unique",             # Column combination must be unique
    "row_count",          # Row count within [min, max]
    "accepted_values",    # Column values must be in an allowed list
    "regex",              # Column values match a regex pattern
    "range",              # Numeric column within [min, max]
    "date_range",         # Date column within [min, max]
    "referential",        # FK values exist in a reference table
    "freshness",          # Data updated within N hours
    "custom_sql",         # Arbitrary SQL returning (pass BOOL, row_count LONG)
    "completeness",       # Null rate below a threshold
    "duplicate_count",    # Duplicate row count below a threshold
    # ── New 4 ────────────────────────────────────────────────────────────
    "conditional_not_null",  # Not-null enforced only when a condition is met
    "mutual_exclusivity",    # Exactly one column in a group is populated
    "character_set",         # Text contains only allowed characters
    "length_check",          # String length within [min_length, max_length]
}

VALID_SEVERITIES = {"CRITICAL", "WARNING", "INFO"}

# Required top-level fields per rule type
RULE_REQUIRED_FIELDS = {
    "not_null":             ["column"],
    "unique":               ["columns"],
    "row_count":            [],                   # min and/or max — checked separately
    "accepted_values":      ["column", "values"],
    "regex":                ["column", "pattern"],
    "range":                ["column"],            # min and/or max — checked separately
    "date_range":           ["column"],
    "referential":          ["column", "ref_table", "ref_column"],
    "freshness":            ["column", "max_hours"],
    "custom_sql":           ["sql"],
    "completeness":         ["column", "max_null_rate"],
    "duplicate_count":      ["columns"],
    "conditional_not_null": ["column", "condition_column", "condition_value"],
    "mutual_exclusivity":   ["columns"],
    "character_set":        ["column"],
    "length_check":         ["column"],
}

# Built-in named character sets for the character_set rule
CHARSET_PATTERNS = {
    "ascii_printable":       r"^[\x20-\x7E]*$",
    "alphanumeric":          r"^[A-Za-z0-9]*$",
    "numeric":               r"^[0-9]*$",
    "alpha":                 r"^[A-Za-z]*$",
    "alphanumeric_extended": r"^[A-Za-z0-9 \-_.,]*$",
    "latin1":                r"^[\x20-\x7E\xA0-\xFF]*$",
}

## Helper — Load YAML from a path

In [0]:
def _load_yaml(path):
    """
    Load a YAML file from a local filesystem path or a DBFS path.

    Databricks tip: for DBFS paths (dbfs:/...) use the /dbfs/ prefix so Python
    can open them with a regular file handle, e.g. /dbfs/FileStore/dq/...
    """
    path = str(path)
    try:
        with open(path, "r", encoding="utf-8") as fh:
            data = yaml.safe_load(fh)
        if not isinstance(data, dict):
            raise DQConfigError(f"Config file is not a valid YAML mapping: {path}")
        return data
    except FileNotFoundError:
        raise DQConfigError(
            f"Config file not found: {path}\n"
            f"  • If using DBFS, prefix with /dbfs/ e.g. /dbfs/FileStore/dq-framework/...\n"
            f"  • If using a Workspace repo, use the full /Workspace/... path."
        )
    except yaml.YAMLError as exc:
        raise DQConfigError(f"YAML parse error in {path}:\n  {exc}")


def _resolve_path(base_dir, config_file):
    """Resolve config_file path relative to base_dir."""
    if os.path.isabs(config_file):
        return config_file
    return str(Path(base_dir) / config_file)

## Core — Validate a Single Rule Dict

In [0]:
def _validate_rule(rule, idx, file_path, existing_ids):
    """
    Validate one rule dict from a dataset config.

    Returns a list of error strings (empty list = rule is valid).
    Does NOT raise — errors are collected and raised all at once by the caller.

    Parameters
    ----------
    rule         : dict       the rule definition
    idx          : int        0-based index (for error message numbering)
    file_path    : str        dataset config file path (for error context)
    existing_ids : set[str]  rule IDs already seen in this file (duplicate check)
    """
    errors = []
    prefix = f"  [rule #{idx + 1}]"

    if not isinstance(rule, dict):
        errors.append(f"{prefix} Not a valid YAML mapping: {rule}")
        return errors

    # ── id ────────────────────────────────────────────────────────────────────────
    rule_id = rule.get("id")
    if not rule_id:
        errors.append(f"{prefix} Missing required field 'id'.")
    elif rule_id in existing_ids:
        errors.append(f"{prefix} Duplicate rule id '{rule_id}' — every rule id must be unique within a dataset config.")

    # ── description ───────────────────────────────────────────────────────────────
    if not rule.get("description"):
        errors.append(f"{prefix} (id={rule_id}) Missing 'description'.")

    # ── type ──────────────────────────────────────────────────────────────────────
    rule_type = rule.get("type")
    if not rule_type:
        errors.append(f"{prefix} (id={rule_id}) Missing 'type'.")
    elif rule_type not in VALID_RULE_TYPES:
        errors.append(
            f"{prefix} (id={rule_id}) Invalid rule type '{rule_type}'.\n"
            f"    Allowed types: {sorted(VALID_RULE_TYPES)}"
        )
    else:
        # ── Required fields for this type ──────────────────────────────────────────
        for field_name in RULE_REQUIRED_FIELDS.get(rule_type, []):
            if field_name not in rule:
                errors.append(
                    f"{prefix} (id={rule_id}) Rule type '{rule_type}' requires field '{field_name}'."
                )

        # ── Type-specific extra validations ────────────────────────────────────────

        # row_count / range: need at least one bound
        if rule_type in ("row_count", "range"):
            if "min" not in rule and "max" not in rule:
                errors.append(
                    f"{prefix} (id={rule_id}) '{rule_type}' needs at least one of 'min' or 'max'."
                )

        # completeness: max_null_rate must be 0.0–1.0
        if rule_type == "completeness":
            rate = rule.get("max_null_rate")
            if rate is not None:
                try:
                    if not (0.0 <= float(rate) <= 1.0):
                        errors.append(
                            f"{prefix} (id={rule_id}) 'max_null_rate' must be between 0.0 and 1.0, got {rate}."
                        )
                except (TypeError, ValueError):
                    errors.append(f"{prefix} (id={rule_id}) 'max_null_rate' must be a number, got {rate!r}.")

        # conditional_not_null: operator validation + list enforcement
        if rule_type == "conditional_not_null":
            valid_ops = {"eq", "ne", "gt", "gte", "lt", "lte", "in", "not_in", "is_null", "is_not_null"}
            op = rule.get("condition_operator", "eq")
            if op not in valid_ops:
                errors.append(
                    f"{prefix} (id={rule_id}) 'condition_operator' must be one of {sorted(valid_ops)}, got '{op}'."
                )
            if op in ("in", "not_in"):
                cv = rule.get("condition_value")
                if cv is not None and not isinstance(cv, list):
                    errors.append(
                        f"{prefix} (id={rule_id}) 'condition_value' must be a list when "
                        f"condition_operator is '{op}', got {type(cv).__name__}."
                    )
            # is_null / is_not_null don't need condition_value — remove any false required-field error
            if op in ("is_null", "is_not_null") and "condition_value" not in rule:
                errors = [
                    e for e in errors
                    if f"(id={rule_id}) Rule type 'conditional_not_null' requires field 'condition_value'" not in e
                ]

        # mutual_exclusivity: needs ≥ 2 columns
        if rule_type == "mutual_exclusivity":
            cols = rule.get("columns", [])
            if isinstance(cols, list) and len(cols) < 2:
                errors.append(
                    f"{prefix} (id={rule_id}) 'mutual_exclusivity' requires at least 2 columns, got {len(cols)}."
                )

        # character_set: exactly one of charset_name / allowed_chars
        if rule_type == "character_set":
            has_chars = "allowed_chars" in rule
            has_name  = "charset_name"  in rule
            if not has_chars and not has_name:
                errors.append(
                    f"{prefix} (id={rule_id}) 'character_set' requires either 'charset_name' "
                    f"(one of {sorted(CHARSET_PATTERNS)}) or 'allowed_chars' (explicit string)."
                )
            if has_chars and has_name:
                errors.append(
                    f"{prefix} (id={rule_id}) 'character_set' must specify 'charset_name' OR "
                    f"'allowed_chars', not both."
                )
            if has_name:
                cn = rule.get("charset_name", "")
                if cn not in CHARSET_PATTERNS:
                    errors.append(
                        f"{prefix} (id={rule_id}) Unknown 'charset_name': '{cn}'. "
                        f"Allowed: {sorted(CHARSET_PATTERNS)}."
                    )

        # length_check: at least one bound, both must be non-negative integers
        if rule_type == "length_check":
            min_len = rule.get("min_length")
            max_len = rule.get("max_length")
            if min_len is None and max_len is None:
                errors.append(
                    f"{prefix} (id={rule_id}) 'length_check' requires at least one of "
                    f"'min_length' or 'max_length'."
                )
            for bound_name, bound_val in (("min_length", min_len), ("max_length", max_len)):
                if bound_val is not None:
                    is_valid = (
                        isinstance(bound_val, int)
                        and not isinstance(bound_val, bool)
                        and bound_val >= 0
                    )
                    if not is_valid:
                        errors.append(
                            f"{prefix} (id={rule_id}) '{bound_name}' must be a non-negative integer, got {bound_val!r}."
                        )

    # ── severity ──────────────────────────────────────────────────────────────────
    severity = rule.get("severity")
    if not severity:
        errors.append(f"{prefix} (id={rule_id}) Missing 'severity'.")
    elif severity not in VALID_SEVERITIES:
        errors.append(
            f"{prefix} (id={rule_id}) Invalid severity '{severity}'. "
            f"Allowed: {sorted(VALID_SEVERITIES)}."
        )

    return errors

## Core — Load and Validate Master Config

In [0]:
def load_master_config(master_path):
    """
    Load master_config.yml and validate its structure.

    Checks performed:
      1. File must exist and be valid YAML
      2. Top-level 'datasets' key must be a non-empty list
      3. Every entry must have 'name', 'config_file', and 'enabled'
      4. Dataset names must be unique (NO duplicates)

    Parameters
    ----------
    master_path : str
        Full path to master_config.yml.
        Use /dbfs/... for DBFS, /Workspace/... for Repo paths.

    Returns
    -------
    dict with keys: 'datasets' (list), 'global_settings' (dict)

    Raises
    ------
    DQConfigError on any structural or duplicate-name issue.
    """
    raw = _load_yaml(master_path)

    # ── 'datasets' key must exist and be a non-empty list ─────────────────────
    if "datasets" not in raw:
        raise DQConfigError(
            f"master_config.yml is missing the required top-level key 'datasets'.\n"
            f"  File: {master_path}"
        )
    datasets_raw = raw["datasets"]
    if not isinstance(datasets_raw, list) or len(datasets_raw) == 0:
        raise DQConfigError("'datasets' in master_config.yml must be a non-empty list.")

    # ── Every entry must have required fields ───────────────────────────────
    entry_errors = []
    for i, entry in enumerate(datasets_raw):
        if not isinstance(entry, dict):
            entry_errors.append(f"  [entry #{i+1}] Not a valid mapping: {entry}")
            continue
        for required_field in ("name", "config_file", "enabled"):
            if required_field not in entry:
                entry_errors.append(
                    f"  [entry #{i+1}] Missing field '{required_field}' "
                    f"(name={entry.get('name', '<unknown>')})"
                )
    if entry_errors:
        raise DQConfigError(
            "master_config.yml has structural errors:\n" + "\n".join(entry_errors)
        )

    # ── Duplicate name check ──────────────────────────────────────────────────
    all_names = [e["name"] for e in datasets_raw]
    seen      = set()
    dupes     = []
    for name in all_names:
        if name in seen:
            dupes.append(name)
        seen.add(name)

    if dupes:
        dup_str = ", ".join(f"'{d}'" for d in sorted(set(dupes)))
        raise DQConfigError(
            f"DUPLICATE DATASET NAMES detected in master_config.yml: {dup_str}\n"
            f"Each dataset name must be unique. Fix duplicates before proceeding."
        )

    return {
        "datasets":        datasets_raw,
        "global_settings": raw.get("global_settings", {}),
    }

## Core — Load and Validate a Single Dataset Config

In [0]:
def load_dataset_config(entry, base_dir):
    """
    Load and validate one dataset config file referenced from master_config.

    Checks performed:
      1. config_file must exist on disk
      2. File must be valid YAML
      3. Top-level 'dataset' block must exist and have 'name' and 'source.path'
      4. dataset.name must EXACTLY match the name in master_config
      5. 'rules' block must exist and have at least one rule
      6. Every rule is validated (type, severity, required fields, type-specific logic)

    Parameters
    ----------
    entry    : dict   one entry from master_config 'datasets' list
    base_dir : str    repo root — used to resolve relative config_file paths

    Returns
    -------
    Parsed dataset config dict (with '_master_entry' key attached).

    Raises
    ------
    DQDatasetConfigError with ALL errors for this dataset listed together.
    """
    master_name  = entry["name"]
    config_file  = entry["config_file"]
    resolved     = _resolve_path(base_dir, config_file)
    errors       = []

    # ── 1. File must exist ──────────────────────────────────────────────────────
    if not os.path.isfile(resolved):
        raise DQDatasetConfigError(
            f"Dataset '{master_name}': config_file not found on disk.\n"
            f"  Expected path : {resolved}\n"
            f"  Check:\n"
            f"    • The path in master_config.yml is correct\n"
            f"    • For DBFS paths, prefix with /dbfs/ (e.g. /dbfs/FileStore/dq/...)\n"
            f"    • For Repo paths, use /Workspace/Repos/.../config/datasets/..."
        )

    # ── 2. Load YAML ──────────────────────────────────────────────────────────
    raw = _load_yaml(resolved)

    # ── 3. dataset block ────────────────────────────────────────────────────────
    if "dataset" not in raw:
        raise DQDatasetConfigError(
            f"Dataset '{master_name}': missing top-level 'dataset' key in {resolved}."
        )
    ds_block = raw["dataset"]
    if not isinstance(ds_block, dict):
        raise DQDatasetConfigError(
            f"Dataset '{master_name}': 'dataset' key must be a YAML mapping in {resolved}."
        )

    # ── 4. name must match master ─────────────────────────────────────────────────
    config_name = ds_block.get("name")
    if config_name is None:
        errors.append(f"  'dataset.name' is missing inside {resolved}.")
    elif config_name != master_name:
        errors.append(
            f"  NAME MISMATCH:\n"
            f"    master_config.yml lists name = '{master_name}'\n"
            f"    dataset.name inside {resolved} = '{config_name}'\n"
            f"    These must be identical."
        )

    # ── 5. source.path must exist ─────────────────────────────────────────────────
    source = ds_block.get("source", {})
    if not isinstance(source, dict) or "path" not in source:
        errors.append(f"  'dataset.source.path' is missing in {resolved}.")

    # ── 6. rules block must exist and have at least one rule ───────────────────
    rules = raw.get("rules")
    if not rules or not isinstance(rules, list):
        errors.append(
            f"  'rules' block is missing or empty in {resolved}.\n"
            f"  At least one rule is required."
        )
    else:
        rule_ids = set()
        for j, rule in enumerate(rules):
            rule_errors = _validate_rule(rule, j, resolved, rule_ids)
            errors.extend(rule_errors)
            if isinstance(rule, dict) and "id" in rule:
                rule_ids.add(rule["id"])

    # ── Raise with ALL errors at once ───────────────────────────────────────────
    if errors:
        raise DQDatasetConfigError(
            f"Dataset '{master_name}' config validation FAILED ({resolved}):\n"
            + "\n".join(errors)
        )

    raw["_master_entry"] = entry
    return raw

## Orchestrator — Load All Configs (Master + All Enabled Datasets)

In [0]:
def load_all_configs(master_path, base_dir=None):
    """
    Full config pre-flight: load master + all enabled dataset configs.

    ALL errors across ALL datasets are collected before raising —
    so you see every problem in one run, not one error at a time.

    Parameters
    ----------
    master_path : str   path to master_config.yml
    base_dir    : str   repo root (auto-detected from master_path if None)

    Returns
    -------
    (dataset_configs, global_settings)
      dataset_configs : list of parsed dataset config dicts (enabled datasets only)
      global_settings : dict of global DQ settings from master_config

    Raises
    ------
    DQConfigError — aborts the run if any config is invalid.
    """
    master_path = str(master_path)
    if base_dir is None:
        # master_config.yml lives at <repo_root>/config/master_config.yml
        # so go up two levels to reach repo root
        base_dir = str(Path(master_path).parent.parent)

    # ── Step 1: load and validate master ─────────────────────────────────────
    master      = load_master_config(master_path)
    all_entries = master["datasets"]
    global_cfg  = master["global_settings"]

    enabled_entries  = [e for e in all_entries if e.get("enabled", True)]
    disabled_entries = [e for e in all_entries if not e.get("enabled", True)]

    if disabled_entries:
        disabled_names = [e["name"] for e in disabled_entries]
        print(f"[CONFIG] Skipping disabled datasets: {disabled_names}")

    if not enabled_entries:
        raise DQConfigError(
            "No enabled datasets found in master_config.yml.\n"
            "Set at least one dataset entry to 'enabled: true'."
        )

    # ── Step 2: load each enabled dataset config — collect ALL errors ─────────
    dataset_configs = []
    dataset_errors  = []

    for entry in enabled_entries:
        try:
            cfg = load_dataset_config(entry, base_dir)
            dataset_configs.append(cfg)
            print(f"[CONFIG] ✅  Loaded: {entry['name']}  ({entry['config_file']})")
        except DQDatasetConfigError as exc:
            dataset_errors.append(str(exc))

    # ── Step 3: raise ALL errors together ────────────────────────────────────
    if dataset_errors:
        sep        = "=" * 70
        error_body = f"\n\n{sep}\n\n".join(dataset_errors)
        raise DQConfigError(
            f"CONFIG VALIDATION FAILED — {len(dataset_errors)} dataset(s) have errors.\n"
            f"Fix ALL errors below before re-running the DQ controller.\n\n"
            f"{sep}\n{error_body}\n{sep}"
        )

    print(
        f"[CONFIG] ✅  All {len(dataset_configs)} enabled dataset config(s) validated successfully."
    )
    return dataset_configs, global_cfg

### ✅ Config validator notebook loaded

This notebook defines:
* `DQConfigError`, `DQDatasetConfigError`
* `VALID_RULE_TYPES`, `VALID_SEVERITIES`, `RULE_REQUIRED_FIELDS`, `CHARSET_PATTERNS`
* `load_all_configs()`, `load_master_config()`, `load_dataset_config()`, `_validate_rule()`

All symbols are available in the calling notebook after `%run`.